# Chapter 02: Text data as model-ready tensors


A language model never receives text directly. A tokenizer maps text to
integer IDs, an embedding table maps IDs to vectors, and a sampling
pipeline groups IDs into input/target windows. These exercises isolate
two easy-to-miss boundaries: bytes are not characters, and next-token
targets are shifted views of the same token stream.


    This notebook is an independently written practice companion. It
    does not reproduce the book's prose, figures, data, or solutions.
    Read the [Chinese chapter](https://github.com/skindhu/Build-A-Large-Language-Model-CN/blob/main/cn-Book/2.%E5%A4%84%E7%90%86%E6%96%87%E6%9C%AC%E6%95%B0%E6%8D%AE.md) for the complete narrative
    and exercise statements, and use the
    [canonical MIT companion code](https://github.com/rasbt/LLMs-from-scratch)
    as an additional reference.

    **Prerequisites:** Python 3.10+ and PyTorch 2.2+. Every example is
    synthetic, deterministic, CPU-friendly, and makes no network call.

    **Workflow**

    1. Read the concept and trace the shapes by hand.
    2. Replace one `LLM_TODO` and its `NotImplementedError`.
    3. Run that exercise's check cell.
    4. Explain the result before moving to the next exercise.
    5. Compare with `solution.py` only after making a serious attempt.

In [ ]:
from __future__ import annotations

from collections.abc import Mapping, Sequence

import torch
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

## Exercise 2.1: Reconstruct byte-token pieces

    **Objective.** Join token byte pieces in ID order and decode the complete stream once.


UTF-8 uses a variable number of bytes per character. A tokenizer may
store a byte sequence in pieces whose boundaries do not align with
Unicode character boundaries. Decoding each piece independently can
therefore fail even when their concatenation is valid text.

Treat tokenization as two mappings: token IDs select byte strings, then
the concatenated byte string is decoded. If IDs are
$[t_0,\ldots,t_n]$ and $V(t)$ returns bytes, the reconstructed stream is
$B=V(t_0)\Vert\cdots\Vert V(t_n)$. Only then compute
`B.decode("utf-8")`.


    ### Data-flow walkthrough


In the toy vocabulary, the Chinese character is split across two token
pieces: `b"\xe4"` and `b"\xbd\xa0"`. Neither is a complete UTF-8
character alone, but their concatenation is `b"\xe4\xbd\xa0"`.
Preserve token order, join as bytes, and decode the final sequence.


    ### Hints and common mistakes


Do not call `.decode()` inside the loop. Surface an unknown token ID as
a useful `ValueError` rather than silently dropping it. Joining Python
strings is also wrong here because the vocabulary values are bytes.

In [ ]:
def reconstruct_text(
    token_ids: Sequence[int], vocab: Mapping[int, bytes]
) -> str:
    # LLM_TODO_CH02_01_BYTES
    raise NotImplementedError(
        "join all byte pieces before UTF-8 decoding"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
pieces = {0: b"A", 1: b"\xe4", 2: b"\xbd\xa0", 3: b"!"}
assert reconstruct_text([0, 1, 2, 3], pieces) == "A\u4f60!"
try:
    reconstruct_text([99], pieces)
except ValueError:
    pass
else:
    raise AssertionError("unknown IDs should be reported")
print("Exercise 2.1 passed")

## Exercise 2.2: Create shifted sliding windows

    **Objective.** Build deterministic input/target windows and verify their shape and stride.


Autoregressive training asks the model to predict the next token at
every position. For a token stream $x_0,x_1,\ldots$, a window beginning
at $s$ uses input $[x_s,\ldots,x_{s+L-1}]$ and target
$[x_{s+1},\ldots,x_{s+L}]$. The two tensors have the same length, but
the target is shifted by one.

`stride` controls the distance between starting positions, not the
distance between tokens inside one window. A stride smaller than
`max_length` creates overlapping examples; a larger stride skips parts
of the stream. Only complete input/target pairs should be emitted.


    ### Data-flow walkthrough


For IDs `0..11`, `max_length=4`, and `stride=3`, starts are `0, 3, 6`.
The first two inputs in a batch of two are `[0,1,2,3]` and
`[3,4,5,6]`; their targets are `[1,2,3,4]` and `[4,5,6,7]`.
The first column therefore reveals the start-position stride.


    ### Hints and common mistakes


The range must leave one extra token for the shifted target. Validate
positive lengths and strides. Keep `shuffle=False` for this reasoning
exercise so batch order remains inspectable.

In [ ]:
class SlidingWindowDataset(Dataset[tuple[Tensor, Tensor]]):
    def __init__(
        self, token_ids: Sequence[int], max_length: int, stride: int
    ) -> None:
        # LLM_TODO_CH02_02_WINDOWS
        raise NotImplementedError("construct complete shifted windows")

    def __len__(self) -> int:
        raise NotImplementedError("return the window count")

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor]:
        raise NotImplementedError("return one input/target pair")


def make_loader(
    token_ids: Sequence[int],
    max_length: int,
    stride: int,
    batch_size: int,
) -> DataLoader[tuple[Tensor, Tensor]]:
    raise NotImplementedError("create a deterministic DataLoader")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
loader = make_loader(range(12), max_length=4, stride=3, batch_size=2)
inputs, targets = next(iter(loader))
assert inputs.shape == targets.shape == (2, 4)
assert inputs[:, 0].tolist() == [0, 3]
assert torch.equal(targets[:, :-1], inputs[:, 1:])
print("Exercise 2.2 passed")

## Chapter 02 synthesis


- Why can a valid character fail to decode when a token piece is viewed alone?
- How does decreasing stride change data reuse and training cost?
- Why does a length-$L$ input require $L+1$ source tokens?


    After answering these questions, run the chapter's `solution.py`
    from a terminal to compare behavior. The reference file is compact
    by design; the reasoning in this notebook is the main lesson.